- 19 UKBN2 stations held out (stratified by BFI tercile); 127 used for training
- Models: persistence, climatology, Ridge SARX, LightGBM; all evaluated on held-out test (2020-2024) in log space
- Outputs: `results/spatial_holdout_results.csv`, `../figures-log/spatial_holdout_bfi_nse.png`

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../figures-log', exist_ok=True)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

HORIZONS = [1, 3, 7, 10, 14]
ALPHA    = 0.1    # Ridge regularisation — from main notebook's CV (best_alpha=0.10)
N_HOLDOUT = 20
SEED     = 42
CLIP_LOW = 1e-4

In [2]:
train_data = pd.read_feather('../checkpoints/train_data.feather')
val_data   = pd.read_feather('../checkpoints/val_data.feather')
test_data  = pd.read_feather('../checkpoints/test_data.feather')

for df in [train_data, val_data, test_data]:
    df['date'] = pd.to_datetime(df['date'])

benchmark    = pd.read_csv('../data/benchmark.csv')
benchmark_ids = set(benchmark['Station'].astype(int).tolist())

train_data = train_data[train_data['station_id'].isin(benchmark_ids)].reset_index(drop=True)
val_data   = val_data[val_data['station_id'].isin(benchmark_ids)].reset_index(drop=True)
test_data  = test_data[test_data['station_id'].isin(benchmark_ids)].reset_index(drop=True)

print(f'UKBN2 stations — train: {train_data["station_id"].nunique()}, '
      f'val: {val_data["station_id"].nunique()}, test: {test_data["station_id"].nunique()}')

LAGS = [1, 3, 7, 10, 14]
feature_cols = (
    ['flow', 'rainfall'] +
    [f'discharge_lag{i}' for i in LAGS] +
    [f'precip_lag{i}'    for i in LAGS] +
    ['month_sin', 'month_cos', 'bfi', 'area', 'saar']
)
print(f'Features ({len(feature_cols)}): {feature_cols}')

UKBN2 stations — train: 146, val: 146, test: 146
Features (17): ['flow', 'rainfall', 'discharge_lag1', 'discharge_lag3', 'discharge_lag7', 'discharge_lag10', 'discharge_lag14', 'precip_lag1', 'precip_lag3', 'precip_lag7', 'precip_lag10', 'precip_lag14', 'month_sin', 'month_cos', 'bfi', 'area', 'saar']


In [3]:
bfi_map = train_data.groupby('station_id')['bfi'].first()
bfi_df  = bfi_map.reset_index().rename(columns={'bfi': 'bfi'})
bfi_df  = bfi_df.dropna(subset=['bfi'])

bfi_df['tercile'] = pd.qcut(bfi_df['bfi'], 3, labels=['low', 'mid', 'high'])
rng = np.random.RandomState(SEED)

holdout_ids = []
per_tercile = N_HOLDOUT // 3
for t in ['low', 'mid', 'high']:
    pool = bfi_df[bfi_df['tercile'] == t]['station_id'].values
    n = per_tercile + (1 if t == 'high' and N_HOLDOUT % 3 != 0 else 0)
    chosen = rng.choice(pool, size=min(n, len(pool)), replace=False)
    holdout_ids.extend(chosen.tolist())

holdout_ids = set(holdout_ids)
seen_ids    = set(train_data['station_id'].unique()) - holdout_ids

print(f'Hold-out stations : {len(holdout_ids)}')
print(f'Seen (training)   : {len(seen_ids)}')
holdout_bfi = bfi_df[bfi_df['station_id'].isin(holdout_ids)].set_index('station_id')['bfi']
print(f'Hold-out BFI range: {holdout_bfi.min():.3f} – {holdout_bfi.max():.3f} '
      f'(median {holdout_bfi.median():.3f})')

train_seen = train_data[train_data['station_id'].isin(seen_ids)].reset_index(drop=True)
val_seen   = val_data[val_data['station_id'].isin(seen_ids)].reset_index(drop=True)
test_holdout = test_data[test_data['station_id'].isin(holdout_ids)].reset_index(drop=True)
train_holdout = train_data[train_data['station_id'].isin(holdout_ids)].reset_index(drop=True)

print(f'\ntrain_seen   : {train_seen.shape}')
print(f'val_seen     : {val_seen.shape}')
print(f'test_holdout : {test_holdout.shape}')

Hold-out stations : 19
Seen (training)   : 127
Hold-out BFI range: 0.302 – 0.918 (median 0.427)



train_seen   : (1113282, 42)
val_seen     : (92710, 42)
test_holdout : (32691, 42)


In [4]:
def nse(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

def kge(y_true, y_pred):
    if len(y_true) < 4: return np.nan
    r = np.corrcoef(y_true, y_pred)[0, 1]
    alpha = np.std(y_pred) / np.std(y_true)
    beta  = np.mean(y_pred) / np.mean(y_true)
    return 1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [5]:
clim = (
    train_holdout.assign(log_flow=lambda d: np.log1p(d['flow']))
    .groupby(['station_id', 'month'])['log_flow']
    .mean()
    .rename('clim_mean_log')
    .reset_index()
)

naive_rows = []
for sid, grp in test_holdout.groupby('station_id'):
    g = grp.sort_values('date').copy()
    flow_arr = g['flow'].values
    log_flow_arr = np.log1p(flow_arr)
    month_arr = g['month'].values
    clim_sid  = clim[clim['station_id'] == sid].set_index('month')['clim_mean_log'].to_dict()

    for h in HORIZONS:
        n = len(flow_arr)
        if n <= h: continue
        y_true = log_flow_arr[h:]          # log1p(flow(t+h))
        y_pers = log_flow_arr[:n-h]        # log1p(flow(t)) — persistence
        y_clim = np.array([clim_sid.get(m, np.nan) for m in month_arr[h:]])

        mask = ~(np.isnan(y_true) | np.isnan(y_pers))
        if mask.sum() < 10: continue
        naive_rows.append({
            'station_id': sid, 'horizon': f't+{h}', 'model': 'persistence',
            'holdout_nse': nse(y_true[mask], y_pers[mask]),
            'holdout_kge': kge(y_true[mask], y_pers[mask]),
            'holdout_rmse': rmse(y_true[mask], y_pers[mask]),
            'bfi': holdout_bfi.get(sid, np.nan)
        })

        mask2 = ~(np.isnan(y_true) | np.isnan(y_clim))
        if mask2.sum() < 10: continue
        naive_rows.append({
            'station_id': sid, 'horizon': f't+{h}', 'model': 'climatology',
            'holdout_nse': nse(y_true[mask2], y_clim[mask2]),
            'holdout_kge': kge(y_true[mask2], y_clim[mask2]),
            'holdout_rmse': rmse(y_true[mask2], y_clim[mask2]),
            'bfi': holdout_bfi.get(sid, np.nan)
        })

naive_df = pd.DataFrame(naive_rows)
print('Naive baselines — hold-out median log-NSE per horizon:')
for h in HORIZONS:
    for m in ['persistence', 'climatology']:
        vals = naive_df[(naive_df['horizon']==f't+{h}') & (naive_df['model']==m)]['holdout_nse']
        print(f'  t+{h:>2} {m:>12}: {vals.median():.4f}  (n={len(vals)})')

Naive baselines — hold-out median log-NSE per horizon:
  t+ 1  persistence: 0.7563  (n=18)
  t+ 1  climatology: 0.3493  (n=18)
  t+ 3  persistence: 0.4343  (n=18)
  t+ 3  climatology: 0.3492  (n=18)
  t+ 7  persistence: 0.1809  (n=18)
  t+ 7  climatology: 0.3493  (n=18)
  t+10  persistence: 0.0556  (n=18)
  t+10  climatology: 0.3480  (n=18)
  t+14  persistence: -0.0288  (n=18)
  t+14  climatology: 0.3467  (n=18)


In [6]:
print('Training Ridge on seen stations (log1p target, per-station z-scoring)...')

_scalers = {}
_means   = {}
_stds    = {}
for sid, grp in train_seen.groupby('station_id'):
    tr = grp.dropna(subset=feature_cols)
    if len(tr) < 30: continue
    sc = StandardScaler().fit(tr[feature_cols])
    log_flow = np.log1p(tr['flow'])
    _scalers[sid] = sc
    _means[sid]   = float(log_flow.mean())
    _stds[sid]    = float(log_flow.std())

holdout_scalers = {}
holdout_means   = {}
holdout_stds    = {}
for sid, grp in train_holdout.groupby('station_id'):
    tr = grp.dropna(subset=feature_cols)
    if len(tr) < 10: continue
    log_flow = np.log1p(tr['flow'])
    holdout_scalers[sid] = StandardScaler().fit(tr[feature_cols])
    holdout_means[sid]   = float(log_flow.mean())
    holdout_stds[sid]    = float(log_flow.std())

print(f'  Seen scalers: {len(_scalers)}, Holdout scalers: {len(holdout_scalers)}')

ridge_holdout_rows = []
for h in HORIZONS:
    print(f'  Ridge t+{h}...')
    X_parts, y_parts = [], []
    for sid, sc in _scalers.items():
        tr = train_seen[train_seen['station_id'] == sid].copy()
        tr['target'] = np.log1p(tr['flow'].shift(-h))
        tr = tr.dropna(subset=feature_cols + ['target'])
        if len(tr) < 10: continue
        smean, sstd = _means[sid], _stds[sid]
        if sstd < 1e-8: continue
        X_parts.append(sc.transform(tr[feature_cols]))
        y_parts.append((tr['target'].values - smean) / sstd)  # per-station z-score in log space

    X_tr = np.vstack(X_parts)
    y_tr = np.concatenate(y_parts)
    model = Ridge(alpha=ALPHA).fit(X_tr, y_tr)

    for sid, grp in test_holdout.groupby('station_id'):
        if sid not in holdout_scalers: continue
        smean, sstd = holdout_means[sid], holdout_stds[sid]
        if sstd < 1e-8: continue

        te = grp.copy()
        te['target'] = np.log1p(te['flow'].shift(-h))
        te = te.dropna(subset=feature_cols + ['target'])
        if len(te) < 10: continue

        pred_z = model.predict(holdout_scalers[sid].transform(te[feature_cols]))
        pred   = pred_z * sstd + smean  # back-transform to log space
        y_true = te['target'].values

        ridge_holdout_rows.append({
            'station_id': sid, 'horizon': f't+{h}', 'model': 'ridge',
            'holdout_nse': nse(y_true, pred),
            'holdout_kge': kge(y_true, pred),
            'holdout_rmse': rmse(y_true, pred),
            'bfi': holdout_bfi.get(sid, np.nan)
        })

ridge_ho_df = pd.DataFrame(ridge_holdout_rows)
print('\nRidge (log1p target, per-station z-score) — hold-out median log-NSE per horizon:')
for h in HORIZONS:
    vals = ridge_ho_df[ridge_ho_df['horizon']==f't+{h}']['holdout_nse']
    print(f'  t+{h:>2}: {vals.median():.4f}  (n={len(vals)})')

Training Ridge on seen stations (log1p target, per-station z-scoring)...


  Seen scalers: 127, Holdout scalers: 18
  Ridge t+1...


  Ridge t+3...


  Ridge t+7...


  Ridge t+10...


  Ridge t+14...



Ridge (log1p target, per-station z-score) — hold-out median log-NSE per horizon:
  t+ 1: 0.7753  (n=17)
  t+ 3: 0.5365  (n=17)
  t+ 7: 0.4115  (n=17)
  t+10: 0.3730  (n=17)
  t+14: 0.3593  (n=17)


In [7]:
LGBM_PARAMS = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.05,
    'n_estimators':      1000,
    'num_leaves':        63,
    'min_child_samples': 50,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_lambda':        5.0,
    'reg_alpha':         0.1,
    'n_jobs':            -1,
    'random_state':      SEED,
    'verbose':           -1,
}
EARLY_STOPPING = 50
all_feat = feature_cols + ['station_id']

def build_dataset(df, horizon, feature_cols):
    out = []
    for sid, grp in df.groupby('station_id'):
        g = grp.copy()
        g['target_raw'] = g['flow'].shift(-horizon)
        g = g.dropna(subset=['target_raw'] + feature_cols)
        g = g[g['target_raw'] > 0]
        g['target'] = np.log1p(g['target_raw'])
        out.append(g)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

gbm_holdout_rows = []
for h in HORIZONS:
    print(f'LightGBM t+{h}...')
    tr = build_dataset(train_seen,    h, feature_cols)
    va = build_dataset(val_seen,      h, feature_cols)
    te = build_dataset(test_holdout,  h, feature_cols)
    if tr.empty or va.empty or te.empty:
        print(f'  Skipping t+{h} — insufficient data'); continue

    X_tr, y_tr = tr[all_feat], tr['target']
    X_va, y_va = va[all_feat], va['target']

    model = lgb.LGBMRegressor(**LGBM_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False),
                         lgb.log_evaluation(period=500)])
    best_iter = model.best_iteration_

    te['pred'] = model.predict(te[all_feat], num_iteration=best_iter)
    for sid, grp in te.groupby('station_id'):
        y_t = grp['target'].values       # log1p target
        y_p = grp['pred'].values         # log1p predictions
        if len(y_t) < 10: continue
        gbm_holdout_rows.append({
            'station_id': sid, 'horizon': f't+{h}', 'model': 'gbm',
            'holdout_nse': nse(y_t, y_p),
            'holdout_kge': kge(y_t, y_p),
            'holdout_rmse': rmse(y_t, y_p),
            'bfi': holdout_bfi.get(sid, np.nan)
        })
    vals = pd.DataFrame(gbm_holdout_rows)
    vals = vals[vals['horizon']==f't+{h}']
    print(f'  Median log-NSE: {vals["holdout_nse"].median():.4f}  (n={len(vals)})')

gbm_ho_df = pd.DataFrame(gbm_holdout_rows)

LightGBM t+1...


[500]	valid_0's rmse: 0.228634


[1000]	valid_0's rmse: 0.227522


  Median log-NSE: 0.8675  (n=17)
LightGBM t+3...


  Median log-NSE: 0.6231  (n=17)
LightGBM t+7...


  Median log-NSE: 0.4860  (n=17)
LightGBM t+10...


  Median log-NSE: 0.4107  (n=17)
LightGBM t+14...


  Median log-NSE: 0.3834  (n=17)


In [8]:
all_ho = pd.concat([naive_df, ridge_ho_df, gbm_ho_df], ignore_index=True)

ridge_full = pd.read_csv('results/ridge_multihorizon_results.csv')
gbm_full   = pd.read_csv('results/gbm_multihorizon_results.csv')

print('=' * 75)
print('Spatial Hold-out vs Full-model — Median NSE across ALL horizons')
print('=' * 75)
for model_name, full_df, col in [
    ('Ridge',       ridge_full, 'test_nse'),
    ('GBM',         gbm_full,   'test_nse'),
]:
    print(f'\n  {model_name}:')
    print(f'  {"horizon":>8}  {"hold-out":>10}  {"full-model":>10}  {"drop":>8}')
    print(f'  {"-"*8}  {"-"*10}  {"-"*10}  {"-"*8}')
    for h in HORIZONS:
        ho_vals   = all_ho[(all_ho['model']==model_name.lower()) &
                            (all_ho['horizon']==f't+{h}')]['holdout_nse']
        full_vals = full_df[full_df['horizon']==f't+{h}'][col]
        drop = full_vals.median() - ho_vals.median()
        print(f'  {"t+"+str(h):>8}  {ho_vals.median():>10.4f}  {full_vals.median():>10.4f}  {drop:>+8.4f}')

print()
print('Spearman r(BFI, NSE) on held-out stations:')
for model_name in ['persistence', 'ridge', 'gbm']:
    sub = all_ho[(all_ho['model']==model_name) & (all_ho['horizon']=='t+1')].dropna(subset=['bfi','holdout_nse'])
    if len(sub) < 5: continue
    r, p = spearmanr(sub['bfi'], sub['holdout_nse'])
    print(f'  {model_name:>12} t+1: r_s={r:.3f}  p={p:.4f}  (n={len(sub)})')

Spatial Hold-out vs Full-model — Median NSE across ALL horizons

  Ridge:
   horizon    hold-out  full-model      drop
  --------  ----------  ----------  --------
       t+1      0.7753      0.7490   -0.0264
       t+3      0.5365      0.5082   -0.0284
       t+7      0.4115      0.3814   -0.0300
      t+10      0.3730      0.3267   -0.0464
      t+14      0.3593      0.3122   -0.0471

  GBM:
   horizon    hold-out  full-model      drop
  --------  ----------  ----------  --------
       t+1      0.8675      0.8936   +0.0261
       t+3      0.6231      0.6186   -0.0045
       t+7      0.4860      0.4588   -0.0272
      t+10      0.4107      0.3988   -0.0118
      t+14      0.3834      0.3740   -0.0095

Spearman r(BFI, NSE) on held-out stations:
   persistence t+1: r_s=0.698  p=0.0013  (n=18)
         ridge t+1: r_s=0.775  p=0.0003  (n=17)
           gbm t+1: r_s=0.654  p=0.0044  (n=17)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
model_labels = [
    ('persistence', 'Persistence', '#e07b54'),
    ('ridge',       'Ridge SARX',  'steelblue'),
    ('gbm',         'LightGBM*',   '#2ca02c'),
]

for col_i, (mkey, mlabel, mcolor) in enumerate(model_labels):
    for row_i, h in enumerate([1, 7]):
        ax = axes[row_i, col_i]
        sub = all_ho[(all_ho['model']==mkey) & (all_ho['horizon']==f't+{h}')].dropna(subset=['bfi','holdout_nse'])

        if mkey == 'ridge':
            full_sub = ridge_full[ridge_full['horizon']==f't+{h}'].merge(
                pd.DataFrame({'station_id': list(bfi_map.index), 'bfi': list(bfi_map.values)}),
                on='station_id', how='left'
            ).dropna(subset=['bfi','test_nse'])
            ax.scatter(full_sub['bfi'], full_sub['test_nse'],
                       alpha=0.15, s=15, c='grey', label='Full model (127 seen)', zorder=1)
        elif mkey == 'gbm':
            full_sub = gbm_full[gbm_full['horizon']==f't+{h}'].merge(
                pd.DataFrame({'station_id': list(bfi_map.index), 'bfi': list(bfi_map.values)}),
                on='station_id', how='left'
            ).dropna(subset=['bfi','test_nse'])
            ax.scatter(full_sub['bfi'], full_sub['test_nse'],
                       alpha=0.15, s=15, c='grey', label='Full model (127 seen)', zorder=1)

        ax.scatter(sub['bfi'], sub['holdout_nse'],
                   c=mcolor, s=60, zorder=3, edgecolors='white', linewidths=0.5,
                   label=f'Hold-out (n={len(sub)})')
        ax.axhline(0, color='black', lw=0.8, ls=':')

        r, p = spearmanr(sub['bfi'], sub['holdout_nse']) if len(sub) >= 5 else (np.nan, np.nan)
        ax.set_xlabel('BFI', fontsize=11)
        ax.set_ylabel('NSE (held-out)', fontsize=11)
        ax.set_title(f'{mlabel} — t+{h}  ($r_s={r:.2f}$)', fontsize=11)
        ax.set_xlim(-0.05, 1.05)
        if row_i == 0 and col_i == 0:
            ax.legend(fontsize=9)

fig.suptitle('Spatial Hold-out Validation: BFI vs NSE for 19 stations never seen during training',
             fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('../latex-report/figures', exist_ok=True)
plt.savefig('../figures-log/spatial_holdout_bfi_nse.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [10]:
os.makedirs('results', exist_ok=True)
all_ho.to_csv('results/spatial_holdout_results.csv', index=False)
print(f'Saved {len(all_ho)} rows to results/spatial_holdout_results.csv')
print(f'Columns: {all_ho.columns.tolist()}')
print(f'Models:  {all_ho["model"].unique().tolist()}')
print(f'Stations held out: {sorted(holdout_ids)}')

print('\n=== Final Summary ===')
summary = all_ho.groupby(['model','horizon'])['holdout_nse'].agg(['median','mean','std']).round(4)
print(summary)

Saved 350 rows to results/spatial_holdout_results.csv
Columns: ['station_id', 'horizon', 'model', 'holdout_nse', 'holdout_kge', 'holdout_rmse', 'bfi']
Models:  ['persistence', 'climatology', 'ridge', 'gbm']
Stations held out: [8013, 18001, 26003, 27047, 28046, 33018, 39019, 42008, 53017, 55016, 60012, 65001, 72014, 75017, 83006, 95001, 201008, 203042, 206001]

=== Final Summary ===
                     median    mean     std
model       horizon                        
climatology t+1      0.3493  0.3309  0.1062
            t+10     0.3480  0.3310  0.1061
            t+14     0.3467  0.3299  0.1062
            t+3      0.3492  0.3311  0.1062
            t+7      0.3493  0.3313  0.1062
gbm         t+1      0.8675  0.8453  0.1080
            t+10     0.4107  0.4200  0.2141
            t+14     0.3834  0.3784  0.1927
            t+3      0.6231  0.6285  0.1933
            t+7      0.4860  0.4825  0.2079
persistence t+1      0.7563  0.7628  0.1692
            t+10     0.0556  0.0666  0.4438